# 008 — Datos, evidencia, hipótesis y falsabilidad

Este notebook reutiliza el mismo núcleo ejecutable que `lab.py`. El objetivo no es
ocultar la implementación, sino separar exploración, ejercicio y solución.

**Evidencia esperada:** resultado JSON, interpretación de una decisión y una
limitación documentada.


## 📖 Resumen de la materia

**Falsabilidad (Popper):** una hipótesis es científica si existe una observación posible
que la refutaría. Ninguna cantidad de confirmaciones la *demuestra*; una refutación sólida
la derriba — la asimetría es el motor del método.

Plantilla de hipótesis experimental en IA (todos los componentes son obligatorios):

```text
H: "X supera al baseline B en la métrica M sobre la población D, por al menos δ"
   — sin B no hay comparación; sin D no se sabe a qué generaliza;
   — sin δ el ruido "confirma"; sin M fijada ANTES, hay selección post-hoc.
```

**Jerarquía de evidencia** (débil → fuerte): demo elegida a mano → métrica en
entrenamiento (inválida) → held-out interno → datos externos/otro periodo → replicación
independiente. **Patologías:** leakage (información del test contamina el train),
p-hacking, HARKing, sesgo de publicación, Goodhart.

Ejemplo trabajado: "accuracy 0.99 en fraude" con 1 % de fraude no supera al modelo trivial
"nunca es fraude" (también 0.99). La reformulación falsable exige recall y precisión con
validación temporal — y puede quedar refutada de forma *informativa*.

In [ ]:
from ai_evolution.labs import run_lab
import json

def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


## Solución de referencia

**Ejercicio 1.** (a) No falsable tal cual: "entiende" no tiene definición operativa; se
vuelve falsable al traducirla a una tarea medible. (b) Falsable: métrica, umbral y
población definidos. (c) No falsable: sin plazo ni magnitud, cualquier futuro la
"confirma". (d) Falsable: predicción cuantitativa sobre un experimento de ablación.

**Ejercicio 2.** El primero: las estadísticas de normalización incluyen información del
test (su media y varianza), así que el train "vio" al test. Se inflan las métricas de
generalización, típicamente poco en datasets grandes i.i.d. y mucho en series temporales o
datasets pequeños. La regla: todo lo aprendido (incluida la normalización) se ajusta solo
con train.

**Ejercicio 3.** "Todo negativo": accuracy 0.97, recall 0, precisión indefinida (no
predice positivos). "Todo positivo": accuracy 0.03, recall 1.0, precisión 0.03. Un modelo
real debe superar simultáneamente precisión 0.03 con recall alto — por eso se exige la
curva precisión/recall y no la accuracy de 0.97, que ya la regala el trivial.

**Ejercicio 4.** Lo evaluable es el protocolo: hipótesis escrita antes, con criterio de
refutación explícito; resultado declarado sin reescribir la hipótesis después (anti-HARKing).
Una hipótesis razonable: "con la misma semilla el JSON es idéntico entre corridas"
(corroborable ejecutando dos veces) y "con otra semilla cambian los valores muestreados
pero no las claves del contrato".

In [ ]:
result = run_lab("evaluation", seed=8)
assert result["kind"] == "evaluation"
assert result["evidence"]
show(result)


In [ ]:
# Ejercicio 3 — baselines triviales
n, pos = 1000, 30
neg = n - pos
acc_todo_neg = neg / n
recall_todo_neg = 0.0
acc_todo_pos = pos / n
recall_todo_pos = 1.0
precision_todo_pos = pos / n
print(f"todo-negativo: acc={acc_todo_neg:.2f} recall={recall_todo_neg}")
print(f"todo-positivo: acc={acc_todo_pos:.2f} recall={recall_todo_pos} precision={precision_todo_pos:.2f}")

In [ ]:
# Ejercicio 4 — hipótesis pre-registrada y test de reproducibilidad
# H1: misma semilla → JSON idéntico. H2: otra semilla → mismas claves, valores distintos.
r1a = run_lab("evaluation", seed=1)
r1b = run_lab("evaluation", seed=1)
r2 = run_lab("evaluation", seed=2)
print("H1 (idéntico con misma semilla):", r1a == r1b)
print("H2 (mismas claves con otra semilla):", sorted(r1a) == sorted(r2))
assert r1a == r1b

## Reflexión

1. El JSON del laboratorio separa `evidence` de `limitations`. Explica cómo esa pareja
   implementa el criterio popperiano: ¿qué observación futura refutaría la conclusión de tu
   corrida?
2. Convierte "nuestro chatbot mejora la satisfacción del cliente" en una hipótesis con los
   cuatro componentes (B, M, D, δ). ¿Cuál fue el más difícil de fijar y por qué?
3. ¿En qué se diferencia un experimento exploratorio legítimo de HARKing? Describe el
   protocolo concreto que te protegería al pasar de uno a otro.